# Week 4 — SST-2 + RoBERTa Ablation Study (N=1000)
**Project**: Quantum Kernels on CLIP/BERT Embeddings — Encoder Comparison

**ROBERTA version of SST2 ablations.** Same as your BERT Week 4.
Each kernel matrix saved immediately. Metric = binary F1.

**Ablations:**
1. Feature maps — Angle vs IQP 1L (reused) vs IQP 2L (8 qubits)
2. Qubit count — 4 vs 8 (IQP 1L)
3. C parameter — sweep N=200, apply best at N=1000

**Inputs:** `sst2_roberta_embeddings.npz`, `sst2_roberta_baseline_results.pkl`, `sst2_roberta_kernel_matrices.npz`
**Runtime:** ~4 hrs (2 new kernel matrices + 4-qubit run)

In [ ]:
!pip install pennylane pennylane-lightning --quiet
import pennylane as qml
import numpy as np
import time, json, pickle, os, shutil, warnings
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings('ignore')
print(f"PennyLane {qml.__version__}")
dev4 = qml.device('lightning.qubit', wires=4)
dev8 = qml.device('lightning.qubit', wires=8)

---
## Step 0: Storage + helpers

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
TAG = 'sst2_roberta'
print(f"Save dir: {SAVE_DIR} | Tag: {TAG}")

def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")

def checkpoint(filename, save_fn):
    save_fn(filename); _persist_and_download(filename)

def save_figure(fig, name):
    for ext in ('png','pdf'):
        p = f"{name}.{ext}"
        fig.savefig(p, bbox_inches='tight', dpi=150)
        _persist_and_download(p)

def _find(fname):
    dp = os.path.join(SAVE_DIR, fname)
    return dp if os.path.exists(dp) else fname

print("Helpers ready.")

---
## Step 1: Load data + Week 3 kernel matrix

In [ ]:
data = np.load(_find(f'{TAG}_embeddings.npz'))
wk3  = np.load(_find(f'{TAG}_kernel_matrices.npz'))

X_tr_full = data['X_tr_subset'].astype(np.float64)
X_te_full = data['X_te_subset'].astype(np.float64)
y_tr_full = data['y_tr_subset']
y_te_full = data['y_te_subset']

K_wk3_train = wk3['K_train_quantum']
K_wk3_test  = wk3['K_test_quantum']

with open(_find(f'{TAG}_baseline_results.pkl'), 'rb') as f:
    baselines = pickle.load(f)

N_ABL = 1000
X_tr, y_tr = X_tr_full[:N_ABL], y_tr_full[:N_ABL]
X_te, y_te = X_te_full, y_te_full

def mf1(yt, yp): return float(f1_score(yt, yp, average='binary'))

print(f"Train {X_tr.shape}  Test {X_te.shape}")
print(f"K_wk3 train {K_wk3_train.shape}  test {K_wk3_test.shape}")
assert K_wk3_train.shape == (1000,1000), "Week 3 matrix not N=1000"

best_baseline_f1 = max(v['f1'] for v in baselines.values())
linear_f1 = baselines.get('Linear', {}).get('f1', 0)
print(f"\nBaselines (metric=binary F1):")
for k, v in baselines.items():
    print(f"  {k:<12} acc={v['accuracy']:.4f}  f1={v['f1']:.4f}")

---
## Step 2: Feature maps

In [ ]:
def angle_map(x, wires):
    for i, w in enumerate(wires): qml.RY(x[i], wires=w)

def iqp_1layer(x, wires):
    n = len(wires)
    for w in wires: qml.Hadamard(wires=w)
    for i, w in enumerate(wires): qml.RZ(x[i], wires=w)
    for i in range(n-1):
        qml.CNOT(wires=[wires[i], wires[i+1]])
        qml.RZ(x[i]*x[i+1], wires=wires[i+1])
        qml.CNOT(wires=[wires[i], wires[i+1]])

def iqp_2layer(x, wires):
    for _ in range(2): iqp_1layer(x, wires)

def make_kernel(fmap, device, n_wires):
    @qml.qnode(device)
    def circ(x1, x2):
        fmap(x1, wires=range(n_wires))
        qml.adjoint(fmap)(x2, wires=range(n_wires))
        return qml.probs(wires=range(n_wires))
    return lambda a, b: float(circ(a, b)[0])

k_angle   = make_kernel(angle_map,  dev8, 8)
k_iqp1    = make_kernel(iqp_1layer, dev8, 8)
k_iqp2    = make_kernel(iqp_2layer, dev8, 8)
k_iqp1_4q = make_kernel(iqp_1layer, dev4, 4)

print("Sanity (K(x,x) must be 1.0):")
for nm, kf, x in [('Angle',k_angle,X_tr[0]),('IQP1L',k_iqp1,X_tr[0]),
                  ('IQP2L',k_iqp2,X_tr[0]),('IQP1L-4q',k_iqp1_4q,X_tr[0,:4])]:
    print(f"  {nm:<12} {kf(x,x):.6f}")

---
## Ablation 1: Feature maps (8 qubits, N=1000)
IQP 1L reused. Angle + IQP 2L each ~83 min.

In [ ]:
fm_results = {}

y_w3 = SVC(kernel='precomputed', C=1.0).fit(K_wk3_train, y_tr).predict(K_wk3_test)
fm_results['IQP 1L'] = {'accuracy': float(accuracy_score(y_te, y_w3)),
                         'f1': mf1(y_te, y_w3), 'reused': True}
print(f"IQP 1L (reused): acc={fm_results['IQP 1L']['accuracy']:.4f}  f1={fm_results['IQP 1L']['f1']:.4f}")

for fname, kfn, ftag in [('Angle', k_angle, 'angle'), ('IQP 2L', k_iqp2, 'iqp2l')]:
    print(f"\nComputing {fname} (N={N_ABL}, ~83 min)...")
    t0 = time.time()
    Ktr = qml.kernels.kernel_matrix(X_tr, X_tr, kfn)
    Kte = qml.kernels.kernel_matrix(X_te, X_tr, kfn)
    print(f"  done {(time.time()-t0)/60:.1f} min — saving:")
    checkpoint(f'{TAG}_kernel_{ftag}_n1000.npz',
               lambda p, a=Ktr, b=Kte: np.savez(p, K_train=a, K_test=b))
    y_p = SVC(kernel='precomputed', C=1.0).fit(Ktr, y_tr).predict(Kte)
    fm_results[fname] = {'accuracy': float(accuracy_score(y_te, y_p)), 'f1': mf1(y_te, y_p)}
    print(f"  {fname}: acc={fm_results[fname]['accuracy']:.4f}  f1={fm_results[fname]['f1']:.4f}")

print(f"\n{'Feature Map':<12} {'Acc':>8} {'F1':>8}")
print("-"*30)
for n, r in fm_results.items():
    print(f"{n:<12} {r['accuracy']:>8.4f} {r['f1']:>8.4f}")

---
## Ablation 2: Qubit count (IQP 1L, N=1000)

In [ ]:
qubit_results = {8: {'accuracy': fm_results['IQP 1L']['accuracy'],
                      'f1': fm_results['IQP 1L']['f1'], 'reused': True}}

print(f"Computing IQP 1L 4-qubit (N={N_ABL}, ~83 min)...")
t0 = time.time()
K4_tr = qml.kernels.kernel_matrix(X_tr[:, :4], X_tr[:, :4], k_iqp1_4q)
K4_te = qml.kernels.kernel_matrix(X_te[:, :4], X_tr[:, :4], k_iqp1_4q)
print(f"  done {(time.time()-t0)/60:.1f} min — saving:")
checkpoint(f'{TAG}_kernel_iqp1l_4q_n1000.npz',
           lambda p: np.savez(p, K_train=K4_tr, K_test=K4_te))
y_p4 = SVC(kernel='precomputed', C=1.0).fit(K4_tr, y_tr).predict(K4_te)
qubit_results[4] = {'accuracy': float(accuracy_score(y_te, y_p4)), 'f1': mf1(y_te, y_p4)}
print(f"  4q: acc={qubit_results[4]['accuracy']:.4f}  f1={qubit_results[4]['f1']:.4f}")

print(f"\n{'Qubits':>7} {'Acc':>8} {'F1':>8}")
for n_q in sorted(qubit_results):
    r = qubit_results[n_q]
    print(f"{n_q:>7} {r['accuracy']:>8.4f} {r['f1']:>8.4f}")
print(f"{'12':>7}   (re-run Week 2 with n_components=12)")

---
## Ablation 3: C parameter (sweep N=200, apply best at N=1000)

In [ ]:
print("C sweep at N=200...")
t0 = time.time()
Kc_tr = qml.kernels.kernel_matrix(X_tr[:200], X_tr[:200], k_iqp1)
Kc_te = qml.kernels.kernel_matrix(X_te, X_tr[:200], k_iqp1)
print(f"  done {time.time()-t0:.0f}s")

C_values = [0.1, 0.5, 1.0, 5.0, 10.0]
c_results = {}
print(f"\n{'C':>6} {'Acc':>8} {'F1':>8}")
print("-"*24)
for C in C_values:
    yp = SVC(kernel='precomputed', C=C).fit(Kc_tr, y_tr[:200]).predict(Kc_te)
    c_results[C] = {'accuracy': float(accuracy_score(y_te, yp)), 'f1': mf1(y_te, yp)}
    print(f"{C:>6.1f} {c_results[C]['accuracy']:>8.4f} {c_results[C]['f1']:>8.4f}")

best_C = max(c_results, key=lambda c: c_results[c]['f1'])
print(f"\nBest C = {best_C} (F1={c_results[best_C]['f1']:.4f})")
if best_C == 1.0:
    print("C=1.0 already used — no re-run needed.")
else:
    yb = SVC(kernel='precomputed', C=best_C).fit(K_wk3_train, y_tr).predict(K_wk3_test)
    print(f"IQP 1L @ C={best_C}, N=1000: acc={accuracy_score(y_te,yb):.4f}  f1={mf1(y_te,yb):.4f}")

---
## Paper Table: SST-2 + RoBERTa ablations

In [ ]:
print("="*64)
print(f"TABLE — SST-2 + RoBERTa (8-dim, N={N_ABL}, metric=binary F1)")
print("="*64)
print(f"{'Method':<22} {'Qubits':>7} {'N':>6} {'Acc':>8} {'F1':>8} {'vsLin':>8}")
print("-"*64)
for name, v in baselines.items():
    print(f"  {'Classical — '+name:<20} {'—':>7} {'1000':>6} "
          f"{v['accuracy']:>8.4f} {v['f1']:>8.4f} {v['f1']-linear_f1:>+8.1%}")
print()
for fname, r in fm_results.items():
    note = ' *' if fname=='IQP 1L' else ''
    print(f"  {'Quantum — '+fname:<20} {'8':>7} {'1000':>6} "
          f"{r['accuracy']:>8.4f} {r['f1']:>8.4f} {r['f1']-linear_f1:>+8.1%}{note}")
print()
for n_q in sorted(qubit_results):
    r = qubit_results[n_q]
    print(f"  {'Quantum — IQP 1L':<20} {n_q:>7} {'1000':>6} "
          f"{r['accuracy']:>8.4f} {r['f1']:>8.4f} {r['f1']-linear_f1:>+8.1%}")
print("="*64)
print("* reused from Week 3")

---
## Figures

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

fm_names = list(fm_results.keys())
fm_f1s = [fm_results[n]['f1'] for n in fm_names]
axes[0].bar(fm_names, fm_f1s, color=['#4D9DE0','#E15554','#3BB273'], width=0.5, edgecolor='white')
axes[0].axhline(best_baseline_f1, color='gray', ls='--', alpha=0.6, label='Best classical')
axes[0].set_ylim(0.4, 1.0); axes[0].set_title(f'Feature maps (8q, N={N_ABL})')
axes[0].set_ylabel('Binary F1'); axes[0].legend(fontsize=9)
for b, v in zip(axes[0].patches, fm_f1s):
    axes[0].text(b.get_x()+b.get_width()/2, v+0.004, f'{v:.3f}', ha='center', fontsize=10, fontweight='500')

q_labels = [f'{n}q' for n in sorted(qubit_results)]+['12q*']
q_f1s = [qubit_results[n]['f1'] for n in sorted(qubit_results)]+[None]
bars2 = axes[1].bar(q_labels, [v if v else 0 for v in q_f1s],
                    color=['#4D9DE0','#E15554','#CCCCCC'], width=0.5, edgecolor='white')
axes[1].axhline(best_baseline_f1, color='gray', ls='--', alpha=0.6, label='Best classical')
axes[1].set_ylim(0.4, 1.0); axes[1].set_title(f'Qubit count (IQP1L, N={N_ABL})')
axes[1].set_ylabel('Binary F1'); axes[1].legend(fontsize=9)
for b, v in zip(bars2, q_f1s):
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+0.004,
                 f'{v:.3f}' if v else 'TBD', ha='center', fontsize=10, fontweight='500')

C_vals = list(c_results.keys())
c_f1s = [c_results[c]['f1'] for c in C_vals]
bi = c_f1s.index(max(c_f1s))
axes[2].bar([str(c) for c in C_vals], c_f1s,
            color=['#E15554' if i==bi else '#4D9DE0' for i in range(len(C_vals))],
            width=0.6, edgecolor='white')
axes[2].axhline(best_baseline_f1, color='gray', ls='--', alpha=0.6, label='Best classical')
axes[2].set_ylim(0.4, 1.0); axes[2].set_title('C sweep (N=200)')
axes[2].set_xlabel('C'); axes[2].set_ylabel('Binary F1'); axes[2].legend(fontsize=9)
for b, v in zip(axes[2].patches, c_f1s):
    axes[2].text(b.get_x()+b.get_width()/2, v+0.004, f'{v:.3f}', ha='center', fontsize=10, fontweight='500')

plt.suptitle('Week 4 Ablations — SST-2 + RoBERTa, N={N_ABL}', y=1.02)
plt.tight_layout(); plt.show()
save_figure(fig, f'{TAG}_fig3_ablations')

---
## Save results

In [ ]:
all_results = {
    'dataset': 'sst2', 'encoder': 'roberta',
    'n_classes': 2, 'metric': 'binary', 'N_ABL': N_ABL,
    'feature_map_ablation': fm_results,
    'qubit_ablation': {str(k): v for k, v in qubit_results.items()},
    'c_sweep': {str(k): v for k, v in c_results.items()},
    'best_C': float(best_C),
    'classical_baselines': baselines,
}
checkpoint(f'{TAG}_results_week4.json',
           lambda p: json.dump(all_results, open(p,'w'), indent=2))

print(f"\n=== WEEK 4 DONE — {TAG} ===")
print(f"Files for {TAG}:")
for f in sorted(os.listdir(SAVE_DIR)):
    if f.startswith(TAG) and ('week4' in f or 'kernel_angle' in f or 'kernel_iqp2l' in f or 'kernel_iqp1l_4q' in f or 'fig3' in f):
        print(f"  {f}")

---
## Week 4 done — SST-2 + RoBERTa

### Saved (tag = `sst2_roberta`):
- `sst2_roberta_kernel_angle_n1000.npz`
- `sst2_roberta_kernel_iqp2l_n1000.npz`
- `sst2_roberta_kernel_iqp1l_4q_n1000.npz`
- `sst2_roberta_fig3_ablations.png/.pdf`
- `sst2_roberta_results_week4.json`

This config's full ablation is complete. Metric = binary F1 throughout.